In [ ]:
#| default_exp core

# aplnb core
> Driving Dyalog APL over the RIDE protocol, and `apl` magics for Jupyter and IPython


In [ ]:
#| export
import atexit,html,json,socket,subprocess
from shutil import which
from importlib.resources import files
from fastcore.utils import *
from fastcore.test import *
from IPython.display import display, Javascript, HTML
from IPython.paths import get_ipython_dir
from IPython.utils.capture import capture_output


aplnb talks to Dyalog using the [RIDE protocol](https://github.com/Dyalog/ride/blob/master/docs/protocol.md), the same TCP protocol used by Dyalog's own IDE and by the official [Jupyter kernel](https://github.com/Dyalog/dyalog-jupyter-kernel). Any Dyalog interpreter can serve it, nothing extra needs to be loaded into the workspace, and the message framing tells us exactly when output ends, whether it was an error, and when the interpreter is ready for more. This notebook builds up the client one step at a time, showing what actually goes over the wire.

## Finding and starting Dyalog

In [ ]:
#| export
def find_dyalog():
    "Locate the Dyalog interpreter binary"
    if p:=which('mapl') or which('dyalog'): return p
    apps = sorted(Path('/Applications').glob('Dyalog-*.app'))
    if apps: return str(apps[-1]/'Contents/Resources/Dyalog/mapl')
    vers = sorted(Path('/opt/mdyalog').glob('*/*/*/mapl'))
    if vers: return str(vers[-1])
    raise FileNotFoundError('Dyalog APL not found: install it from dyalog.com')

On macOS the installer symlinks the interpreter into `/usr/local/bin`, so `which` normally finds it. The fallbacks scan the standard install locations on macOS and Linux.

In [ ]:
find_dyalog()

'/usr/local/bin/dyalog'

RIDE connections can go in either direction. Dyalog's Jupyter kernel uses `RIDE_INIT=SERVE:*:port`, where the interpreter listens and the client polls until the port opens, which means guessing a free port and racing to connect. `CONNECT` mode reverses the roles: we listen on a port the OS just gave us, tell the interpreter to dial back, and `accept()` blocks until it does. No race, no polling. `RIDE_SPAWNED=1` tells the interpreter it belongs to this connection, so it exits when the connection drops.

In [ ]:
#| export
def start_dyalog(
    dyalog=None,  # Path to the interpreter binary; `find_dyalog()` result if None
    timeout=10,   # Socket timeout (secs), so a client bug can never hang the caller
):
    "Spawn a Dyalog interpreter that connects back to us over RIDE; return `(socket,Popen)`"
    if not dyalog: dyalog = find_dyalog()
    lsn = socket.create_server(('127.0.0.1', 0))
    port = lsn.getsockname()[1]
    env = os.environ | dict(RIDE_INIT=f'CONNECT:127.0.0.1:{port}', RIDE_SPAWNED='1',
        MAXAPLCORES=os.environ.get('MAXAPLCORES','0'),
        DYALOGQUIETUCMDBUILD='1', DYALOG_LINEEDITOR_MODE='1', ENABLE_CEF='0', LOG_FILE_INUSE='0')
    dn = subprocess.DEVNULL
    proc = subprocess.Popen([dyalog], env=env, stdin=dn, stdout=dn, stderr=dn)
    lsn.settimeout(10)
    sock,_ = lsn.accept()
    lsn.close()
    sock.settimeout(timeout)
    return sock,proc

In [ ]:
sock,proc = start_dyalog()


## Message framing

The interpreter speaks first. Here are the raw bytes:

In [ ]:
raw = sock.recv(64)
raw

b'\x00\x00\x00\x1cRIDESupportedProtocols=2'

Every message is framed as a 4-byte big-endian total length, the literal `RIDE`, then a UTF-8 payload. Here `0x1c` is 28: 4 length bytes, 4 for `RIDE`, and the 20 characters of `SupportedProtocols=2`. The first two messages in each direction are plain strings negotiating the protocol version; every payload after that is JSON, a 2-element array of command name and arguments. One quirk: the interpreter sometimes sends raw control characters inside JSON strings, which strict parsers reject, so `ride_recv` escapes them before decoding (the official Jupyter kernel does the same).

In [ ]:
#| export
def ride_send(sock, msg):
    "Send one RIDE message: a handshake `str`, or a `[cmd,args]` list sent as JSON"
    if not isinstance(msg,str): msg = json.dumps(msg, separators=(',',':'))
    b = ('RIDE'+msg).encode()
    sock.sendall((len(b)+4).to_bytes(4,'big')+b)

def _recvall(sock, n):
    parts = []
    while n:
        b = sock.recv(n)
        if not b: raise ConnectionError('Dyalog closed the connection')
        parts.append(b)
        n -= len(b)
    return b''.join(parts)

def ride_recv(sock):
    "Receive one RIDE message, JSON-decoded unless it's a handshake string"
    hdr = _recvall(sock, 8)
    assert hdr[4:8]==b'RIDE', f"Bad RIDE header: {hdr}"
    msg = _recvall(sock, int.from_bytes(hdr[:4],'big')-8).decode()
    msg = re.sub(r'[\x00-\x1f]', lambda m: f'\\u{ord(m[0]):04x}', msg)
    return json.loads(msg) if msg[0]=='[' else msg


The handshake is symmetric: each side sends `SupportedProtocols=2` and `UsingProtocol=2`, then we identify ourselves and the interpreter replies with its details:

In [ ]:
ride_send(sock, 'SupportedProtocols=2')
ride_send(sock, 'UsingProtocol=2')
ride_recv(sock)

'UsingProtocol=2'

In [ ]:
ride_send(sock, ['Identify',{'apiVersion':1,'identity':1}])
info = ride_recv(sock)[1]
{k:info[k] for k in ('Vendor','Language','version','arch','platform')}


{'Vendor': 'Dyalog Limited',
 'Language': 'APL',
 'version': '20.0.53963',
 'arch': 'Unicode/64',
 'platform': 'Mac-64'}

The interpreter then sends a series of status messages on its own. `SetPromptType` with `type` 1 means the session is ready for input, so read until that arrives:

In [ ]:
msgs = []
while True:
    m = ride_recv(sock)
    msgs.append(m)
    if m[0]=='SetPromptType' and m[1]['type']==1: break
msgs

[['UpdateDisplayName', {'displayName': 'CLEAR WS'}],
 ['SetPromptType', {'type': 1}]]

## Executing code

`Execute` takes a line of session input (the trailing newline is required). The interpreter echoes the input, streams back session output as `AppendSessionOutput` messages, and finally sends `SetPromptType` 1 to say it's ready for the next line. That final message is what makes RIDE reliable to drive: there's never a question of whether output has finished.

In [ ]:
ride_send(sock, ['Execute',{'text':'3×⍳4\n','trace':0}])
msgs = []
while True:
    m = ride_recv(sock)
    msgs.append(m)
    if m[0]=='SetPromptType' and m[1]['type']==1: break
msgs

[['UpdateSessionCaption', {'text': 'CLEAR WS - Dyalog APL'}],
 ['AppendSessionOutput', {'result': '3×⍳4\n', 'type': 14, 'group': 0}],
 ['SetPromptType', {'type': 0}],
 ['AppendSessionOutput', {'result': '3 6 9 12\n', 'type': 2, 'group': 0}],
 ['SetPromptType', {'type': 1}]]

The `type` field distinguishes kinds of output: 14 is the echo of our own input (clients normally drop it), and the rest are genuine session output. Errors are flagged out-of-band with a `HadError` message, so there's no scraping of output text to detect failure:

In [ ]:
ride_send(sock, ['Execute',{'text':'1÷0\n','trace':0}])
msgs = []
while True:
    m = ride_recv(sock)
    msgs.append(m)
    if m[0]=='SetPromptType' and m[1]['type']==1: break
msgs

[['AppendSessionOutput', {'result': '1÷0\n', 'type': 14, 'group': 0}],
 ['SetPromptType', {'type': 0}],
 ['HadError', {'error': 11, 'dmx': 1}],
 ['AppendSessionOutput',
  {'result': 'DOMAIN ERROR: Divide by zero\n', 'type': 5, 'group': 0}],
 ['AppendSessionOutput', {'result': '      1÷0\n', 'type': 5, 'group': 0}],
 ['AppendSessionOutput', {'result': '       ∧\n', 'type': 5, 'group': 0}],
 ['SetPromptType', {'type': 1}]]

APL errors and unusual prompt states get their own exceptions: `AplError` for ordinary APL errors (carrying the session's error display), and `AplPrompt`, used internally when the session stops at a prompt other than ready.

In [ ]:
#| export
class AplError(Exception): "An APL error, carrying the session's error display as its message"

class AplPrompt(Exception):
    "The session stopped at a non-ready prompt: 2=⎕ input, 3=incomplete input, 4=⍞ input"
    def __init__(self, ptype):
        super().__init__(f'prompt type {ptype}')
        self.ptype = ptype

`ride_run` wraps the whole exchange: normalize the code to non-blank lines, send them all as one `Execute`, drop the echoes and prompt strings, collect everything else, and return the output plus the error number (0 if none) once the session is ready again. The echo counting is the subtle part: each input line is echoed back as a type 14 message, and a prompt change only counts as final once every line we sent has been echoed.


In [ ]:
#| export
def ride_run(sock, code):
    "Run one or more lines of APL in the session; return `(output,errno)` once the session is ready again"
    lines = [l for l in code.splitlines() if l.strip()]
    ride_send(sock, ['Execute',{'text':'\n'.join(lines)+'\n','trace':0}])
    out,err,pending = '',0,len(lines)
    while True:
        m,a = ride_recv(sock)
        if m=='AppendSessionOutput':
            if a['type']==14: pending -= 1
            elif a['type']!=1: out += a['result']
        elif m=='HadError': err = a['error']
        elif m=='SetPromptType' and a['type']!=0 and not pending:
            if a['type']!=1: raise AplPrompt(a['type'])
            return out,err


## Multi-line input

Since 20.0 the session has multiline input on by default, so a complete block construct can go in a single `Execute`. The interpreter echoes each line as it consumes it, switches the prompt to type 3 while a block is open, then runs the block once it's complete. This is why `ride_run` counts echoes: a type 3 prompt seen before all our input has been echoed just means "still reading the block", while one seen after means the input really was incomplete. (One thing to avoid: sending a fresh `Execute` message while the prompt is type 3 crashes the interpreter, so a batch of lines must go in one message.)

In [ ]:
ride_run(sock, '''
:If 1
    ⎕←6×7
:EndIf''')


('42\n', 0)

Plain multi-statement input works the same way, and session state persists across calls:


In [ ]:
test_eq(ride_run(sock, 'x←⍳3\nz←x×x\n⎕←z'), ('1 4 9\n', 0))


Errors come back with the `HadError` number and the session error display:

In [ ]:
out,err = ride_run(sock, '1÷0')
print(out)
test_eq(err, 11)


DOMAIN ERROR: Divide by zero
      1÷0
       ∧



## A session object

`Apl` packages all of the above: spawn the interpreter, shake hands, identify, widen the print width so long results aren't wrapped (the same 32767 the official kernel uses), and wait for the first ready prompt.

In [ ]:
#| export
class Apl:
    "A Dyalog APL session over the RIDE protocol"
    def __init__(self, dyalog=None, timeout=10):
        store_attr()
        self._connect()

    def _connect(self):
        self.sock,self.proc = start_dyalog(self.dyalog, self.timeout)
        assert ride_recv(self.sock)=='SupportedProtocols=2'
        ride_send(self.sock, 'SupportedProtocols=2')
        ride_send(self.sock, 'UsingProtocol=2')
        assert ride_recv(self.sock)=='UsingProtocol=2'
        ride_send(self.sock, ['Identify',{'apiVersion':1,'identity':1}])
        self.info = ride_recv(self.sock)[1]
        ride_send(self.sock, ['SetPW',{'pw':32767}])
        atexit.register(self.close)
        while True:
            m,a = ride_recv(self.sock)
            if m=='SetPromptType' and a['type']==1: break


`close` asks the interpreter to exit with an `Exit` message, then falls back to killing the process: a wedged interpreter ignores both `Exit` and SIGTERM. The interpreter doesn't exit just because the connection drops, so `_connect` registers `close` to run at process exit; without that, every notebook kernel restart would leak a dyalog process.

In [ ]:
#| export
@patch
def close(self:Apl):
    "Shut down the interpreter and close the connection"
    atexit.unregister(self.close)
    try: ride_send(self.sock, ['Exit',{'code':0}])
    except OSError: pass
    self.sock.close()
    try: self.proc.wait(3)
    except subprocess.TimeoutExpired:
        self.proc.kill()
        self.proc.wait()

In [ ]:
apl = Apl()
apl.info['version']

'20.0.53963'

`run` sends the code and interprets what comes back. APL errors raise `AplError` with the session's error display. A `⎕` or `⍞` input request is cancelled (the session survives) and raises. Incomplete input, such as an unclosed `:If`, is the nasty case: it wedges the interpreter beyond recovery ([Dyalog/ride#1401](https://github.com/Dyalog/ride/issues/1401)), so the only honest move is to tell the user, replace the interpreter with a fresh one, and raise. Workspace state is lost when that happens.

In [ ]:
#| export
@patch
def run(self:Apl, code):
    "Run `code` in the session, returning its output; raises `AplError` on APL errors"
    try: out,err = ride_run(self.sock, code)
    except AplPrompt as e:
        if e.ptype in (2,4):
            ride_run(self.sock, '→' if e.ptype==2 else '')
            raise AplError('Input via ⎕ or ⍞ is not supported in aplnb') from None
        print('Incomplete input wedged the Dyalog interpreter; started a fresh session (workspace lost). See Dyalog/ride#1401')
        self.close()
        self._connect()
        raise AplError('Incomplete input') from None
    if err: raise AplError(out)
    return out

In [ ]:
print(apl.run('m←2 3⍴⍳6\n⎕←m'))

1 2 3
4 5 6



Output comes back exactly as the session formats it, so matrices look like APL. State persists for the life of the session, and errors raise `AplError` carrying the session's error display:

In [ ]:
test_eq(apl.run('+/,m'), '21\n')
test_fail(lambda: apl.run('m+\'x\''), contains='DOMAIN ERROR')

Wrapping every use in `print(apl.run(...))` is clunky. Defining `__call__` lets us treat the session itself as a function, and giving the output a verbatim `__repr__` means bare expressions in a notebook display exactly as they do in a Dyalog session. `AplOut` subclasses `str`, so results still compare, slice, and print like ordinary strings. A cell with no output returns None, so nothing displays at all:

In [ ]:
#| export
class AplOut(str):
    "Output text from an `Apl` call; displays verbatim, in the SAX2 APL font where HTML is available"
    def __repr__(self): return str(self)
    def _repr_html_(self): return f'<pre class="aplnb_out sax2">{html.escape(self.rstrip(chr(10)))}</pre>'

@patch
def __call__(self:Apl, code):
    "Run `code`, returning session output (or None if there is none)"
    return AplOut(self.run(code)) or None


In [ ]:
apl('m ∘.× ⍳4')

1  2  3  4
2  4  6  8
3  6  9 12
          
4  8 12 16
5 10 15 20
6 12 18 24

In [ ]:
test_eq(apl('+/,m'), '21\n')
test_is(apl('m2←m×10'), None)

## Getting values into Python

`run` returns display text, which is what you want to look at, but not what you want to compute with. For that, ask the interpreter to serialize the value with `⎕JSON` and parse it on our side. The `HighRank` variant option makes rank≥2 arrays serialize as nested lists (by default `⎕JSON` refuses them). The `1` left argument forces export: monadic `⎕JSON` on a character vector would parse it as JSON instead of serializing it.

In [ ]:
#| export
@patch
def pyval(self:Apl, expr):
    "Evaluate `expr` and return the result as a Python value"
    return json.loads(self.run(f"1(⎕JSON⍠'HighRank' 'Split')({expr})"))

In [ ]:
test_eq(apl.pyval('m'), [[1,2,3],[4,5,6]])
test_eq(apl.pyval('3×⍳4'), [3,6,9,12])
test_eq(apl.pyval('⎕A'), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ')

`pyval` is wordy for the most common case, grabbing a variable. Square brackets read better, and `⎕JSON` works in both directions, so we can assign Python values into the workspace too:

In [ ]:
#| export
def _apljson(v):
    "An APL expression that evaluates to the Python value `v`"
    return "⎕JSON'" + json.dumps(v).replace("'", "''") + "'"

@patch
def __getitem__(self:Apl, expr): return self.pyval(expr)

@patch
def __setitem__(self:Apl, nm, v): self.run(f'{nm}←{_apljson(v)}')

In [ ]:
apl['q'] = [[1,2],[3,4.5]]
test_eq(apl['q'], [[1,2],[3,4.5]])
test_eq(apl["'nested: ',⍕⎕NC'q'"], 'nested: 2')
apl['s'] = "it's"
test_eq(apl['s'], "it's")

Note that `⎕JSON` imports nested lists as vectors of vectors, not as rank-2 arrays. When you want a proper matrix, mix with `↑`:

In [ ]:
apl('↑q')

1 2  
3 4.5

`fn` goes one step further: it turns any APL function into a Python callable. One argument applies it monadically; two applies it dyadically, in APL's left-argument-first order. Arguments travel in as `⎕JSON` and the result comes back through `pyval`:

In [ ]:
#| export
@patch
def fn(self:Apl, code):
    "A Python callable applying APL function `code` monadically or dyadically"
    def f(*args):
        if len(args)==1: return self.pyval(f'({code}){_apljson(args[0])}')
        a,w = args
        return self.pyval(f'({_apljson(a)})({code}){_apljson(w)}')
    return f

In [ ]:
sq = apl.fn('{⍵*2}')
test_eq(sq([1,2,3]), [1,4,9])
test_eq(apl.fn('+/')([1,2,3]), 6)
test_eq(apl.fn('↑')(2, [5,6,7]), [5,6])

Sessions close themselves at process exit, but a context manager is tidier for short-lived ones:

In [ ]:
#| export
@patch
def __enter__(self:Apl): return self

@patch
def __exit__(self:Apl, *args): self.close()

In [ ]:
with Apl() as a2: test_eq(a2('2+2'), '4\n')

## Input requests and incomplete input

A cell that asks for keyboard input can't be satisfied in a notebook, so `run` cancels the request and raises, leaving the session usable:

In [ ]:
test_fail(lambda: apl('x←⎕'), contains='not supported')
test_eq(apl('2+2'), '4\n')

In [ ]:
test_fail(lambda: apl('x←⍞'), contains='not supported')
test_eq(apl('2+2'), '4\n')

Incomplete input is unrecoverable, so `run` replaces the wedged interpreter with a fresh workspace and says so:

In [ ]:
test_fail(lambda: apl(':If 1'), contains='Incomplete')
test_eq(apl('2+2'), '4\n')

Incomplete input wedged the Dyalog interpreter; started a fresh session (workspace lost). See Dyalog/ride#1401


## The `apl` magics

`%%apl` runs a cell and displays the session output, rendered in Adám Brudzewsky's [SAX2](https://github.com/abrudz/SAX2) APL font (loaded from a CDN, falling back to plain monospace offline), so results look exactly as they do in a Dyalog session. `%apl expr` returns the expression's value as a Python object (it's just `apl[expr]`), so it works on the right of an assignment: `z = %apl z`. A trailing `;` on a cell suppresses its output, and the interpreter only starts on first use, not when the magic is registered. The [APL language bar](https://abrudz.github.io/lb/apl) is injected into the page the first time a magic runs.

In [ ]:
#| export
_css = """<style>
@font-face { font-family:'SAX2'; src: local('SAX2'), url('https://cdn.jsdelivr.net/gh/abrudz/SAX2@master/SAX2.ttf') format('truetype') }
.sax2 { font-family:'SAX2',monospace !important; line-height:1.05 !important }
</style>"""

class APLMagic:
    "IPython `%apl`/`%%apl` magics, driving a lazily-started `Apl` session"
    def __init__(self, dyalog=None): self.dyalog,self.o,self._loaded = dyalog,None,False

    def apl(self, line, cell=None):
        "Run APL: a cell magic displays the session output; a line magic returns the expression's Python value"
        if not self.o: self.o = Apl(self.dyalog)
        if not self._loaded:
            display(Javascript((files('aplnb')/'lb.js').read_text()))
            display(HTML(_css))
            self._loaded = True
        if cell is None: return self.o[line.split('⍝')[0].strip()]
        disp,cell = True,cell.rstrip()
        if cell.endswith(';'): disp,cell = False,cell[:-1]
        out = self.o(cell)
        if disp and out: display(out)


In [ ]:
#| export
def create_magic(shell=None):
    "Create an `APLMagic` and register its `apl` line/cell magic with `shell`, returning it"
    if not shell: shell = get_ipython()
    apl_magic = APLMagic()
    shell.register_magic_function(apl_magic.apl, 'line_cell', 'apl')
    return apl_magic


In [ ]:
# Only required if you don't load the extension
magic = create_magic()


In [ ]:
%%apl
m2←3 3⍴⍳9
⎕←m2

Javascript(// APL language bar by Adám Brudzewsky: https://abrudz.github.io/lb (source: https://github.com/abrudz/lb)
// MIT License, Copyright (c) 2011-2020 Nikolay G. Nikolov and Adam Brudzevski. This is a modified copy bundled with aplnb.
// Changes from upstream: double backtick composes ```; insertion via insertText so undo and input events work;
// Monaco editor support (incl. EditContext mode); dark mode; overlay/push-down toggle persisted per site;
// idempotent injection; ResizeObserver-driven layout; @font-face with dead url() removed; skipped on quarto-rendered pages.
; (_ => {
	if (document.querySelector('.ngn_lb')) return
	if (document.querySelector('meta[name=generator][content^=quarto]')) return //no bar on rendered docs pages
	let hc = { '<': '&lt;', '&': '&amp;', "'": '&apos;', '"': '&quot;' }, he = x => x.replace(/[<&'"]/g, c => hc[c]) //html chars and escape fn
		, tcs = '<-←xx×/\\×:-÷*O⍟[-⌹-]⌹OO○77⌈FF⌈ll⌊LL⌊T_⌶II⌶|_⊥TT⊤-|⊣|-⊢=/≠L-≠<=≤<_≤>=≥>_≥==≡=_≡7=≢Z-≢vv∨^^∧^~⍲v~

HTML(<style>
@font-face { font-family:'SAX2'; src: local('SAX2'), url('https://cdn.jsdelivr.net/gh/abrudz/SAX2@master/SAX2.ttf') format('truetype') }
.sax2 { font-family:'SAX2',monospace !important; line-height:1.05 !important }
</style>)

1 2 3
4 5 6
7 8 9

The line magic brings values back into Python:

In [ ]:
z = %apl m2  ⍝ comments are fine here too
test_eq(z, [[1,2,3],[4,5,6],[7,8,9]])

`]` user commands work, and a trailing `;` suppresses cell output entirely:


In [ ]:
%%apl
]display 2 2⍴'ab' 'cd' 1 2

┌→──────────┐
↓ ┌→─┐ ┌→─┐ │
│ │ab│ │cd│ │
│ └──┘ └──┘ │
│           │
│ 1    2    │
│           │
└∊──────────┘

In [ ]:
%%apl
big←1000 1000⍴⍳12;

In [ ]:
#| hide
with capture_output() as cap: magic.apl('', '⍳3;')
test_eq(len(cap.outputs), 0)
with capture_output() as cap: magic.apl('', '⍳3')
test_eq(len(cap.outputs), 1)


In [ ]:
#| export
def load_ipython_extension(ipython):
    "Required function for creating magic"
    create_magic(shell=ipython)

In [ ]:
#| export
def create_ipython_config():
    "Called by `aplnb_install` to install magic"
    ipython_dir = Path(get_ipython_dir())
    cf = ipython_dir/'profile_default'/'ipython_config.py'
    cf.parent.mkdir(parents=True, exist_ok=True)
    if cf.exists() and 'aplnb' in cf.read_text(): return print('aplnb already installed!')
    with cf.open(mode='a') as f: f.write("\nc.InteractiveShellApp.extensions.append('aplnb')\n\n")
    print(f"Jupyter config updated at {cf}")

## Cleanup

Shut down the sessions this notebook started: the magic's, the `Apl` object's, and the raw-socket walkthrough one.

In [ ]:
magic.o.close()
apl.close()
ride_send(sock, ['Exit',{'code':0}])
sock.close()

## Export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()